# HaaS vs local-reduce histogram comparison

Run the Z' → tt̄ single-lepton analysis twice over identical inputs — once with `HistServProcessor` (histograms filled on a remote histserv via gRPC) and once with `HistLocalProcessor` (histograms filled on workers and reduced by coffea). Report side-by-side metrics and verify the final histograms match bin-by-bin.

## AF flag

In [ ]:
AF = "coffeacasa-condor"  # options: [coffeacasa-condor, coffeacasa-gateway, purdue-af-k8s, purdue-af-slurm]
AUTO_CLOSE_CLIENT = False
n_workers = 800

## Imports and dependencies

### The intccms package
Add `src/` and the repo root to `sys.path` so the `intccms` and `example_cms` packages are importable without installation.

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd()
src_dir = repo_root / "src"
examples_dir = repo_root
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
if str(examples_dir) not in sys.path:
    sys.path.insert(0, str(examples_dir))
print(f"✅ Added {src_dir} to Python path")
print(f"✅ Added {examples_dir} to Python path")

### Installing extra dependencies
Install `omegaconf`, `roastcoffea`, and `histserv` on the client. `histserv` is needed so the HaaS backend can talk to the server.

In [ ]:
import subprocess, sys, importlib

def pip_install(spec):
    try:
        subprocess.check_output(
            [sys.executable, "-m", "pip", "install", "-q", spec],
            stderr=subprocess.STDOUT,
        )
        print(f"✅ {spec}")
    except subprocess.CalledProcessError as e:
        print(f"❌ {spec}\n{e.output.decode()}")

def ensure(pkg, spec=None, version=None):
    try:
        mod = importlib.import_module(pkg)
        if version and getattr(mod, "__version__", None) != version:
            raise ImportError
        print(f"✓ {pkg} already installed")
    except ImportError:
        pip_install(spec or pkg)

ensure("omegaconf")
ensure("roastcoffea", "roastcoffea==0.1.2", "0.1.2")
ensure("histserv", "histserv==0.1.9", "0.1.9")

### Alternative coffea version
Pin the coffea version on both the client and the Dask workers.

In [ ]:
COFFEA_VERSION = "2026.4.0"
COFFEA_PIP = COFFEA_VERSION if "git" in COFFEA_VERSION else f"coffea=={COFFEA_VERSION}"
WORKER_DEPENDENCIES = [COFFEA_PIP, "roastcoffea==0.1.2", "histserv==0.1.9"]

pip_install(COFFEA_PIP)

### Imports from stdlib and other libraries

In [ ]:
import cloudpickle
import copy
import time

from coffea.processor import DaskExecutor
from coffea.nanoevents import NanoAODSchema

### Imports from intccms

In [ ]:
from intccms.schema import Config, load_config_with_restricted_cli
from intccms.utils.output import OutputDirectoryManager
from intccms.metadata_extractor import DatasetMetadataManager
from intccms.datasets import DatasetManager
from intccms.analysis import (
    run_processor_workflow,
    HistServProcessor,
    HistLocalProcessor,
)

from roastcoffea import MetricsCollector

### Registering packages with cloudpickle

In [ ]:
import intccms
import example_cms

cloudpickle.register_pickle_by_value(intccms)
cloudpickle.register_pickle_by_value(example_cms)

## Dask client setup

In [ ]:
from intccms.utils.dask_client import acquire_client

## Configuration
Same config for both setups — we want the only difference to be the histogram backend.

In [ ]:
from example_cms.configs.configuration import config as original_config

config = copy.deepcopy(original_config)

config["datasets"]["max_files"] = None  # small, for the comparison loop
config["general"]["output_dir"] = "example_cms_histserv/outputs_full_run_haas_or_not/"
config["general"]["run_metadata_generation"] = True  # reuse cache
config["general"]["run_processor"] = True
config["general"]["run_analysis"] = True
config["general"]["save_skimmed_output"] = False
config["general"]["run_histogramming"] = True
config["general"]["run_systematics"] = False
config["general"]["run_statistics"] = False

full_config = load_config_with_restricted_cli(config, [])
validated_config = Config(**full_config)

## Output manager

In [ ]:
output_manager = OutputDirectoryManager(
    root_output_dir=validated_config.general.output_dir,
    cache_dir=validated_config.general.cache_dir,
    metadata_dir=validated_config.general.metadata_dir,
    skimmed_dir=validated_config.general.skimmed_dir,
)

## Dataset manager and metadata

In [ ]:
dataset_manager = DatasetManager(validated_config.datasets)

metadata_generator = DatasetMetadataManager(
    dataset_manager=dataset_manager,
    output_manager=output_manager,
    config=validated_config,
    chunksize=500_000,
)

if metadata_generator.generate_metadata:
    with acquire_client(
        AF, close_after=AUTO_CLOSE_CLIENT, pip_packages=WORKER_DEPENDENCIES
    , num_workers=n_workers, profile_suffix="full_run_haas_or_not", profile_output_dir=f"{output_manager.root_output_dir}/profiling/") as (client, cluster):
        metadata_generator.run(executor=DaskExecutor(client=client))
else:
    metadata_generator.run()

metadata_lookup = metadata_generator.build_metadata_lookup()
workitems = metadata_generator.workitems
print(f"{len(workitems)} workitems loaded")

## Run both setups
One client context, two processor runs back-to-back. The helper below wraps each run in a `MetricsCollector` so we can compare.

In [ ]:
from collections import defaultdict

def run_backend(client, processor, label, post = None):
    with MetricsCollector(
        client=client,
        processor_instance=processor,
        track_workers=True,
        worker_tracking_interval=1.0,
    ) as collector:
        t0 = time.perf_counter()
        output, report = run_processor_workflow(
            config=validated_config,
            output_manager=output_manager,
            metadata_lookup=metadata_lookup,
            processor=processor,
            workitems=workitems,
            executor=DaskExecutor(client=client, treereduction=8, retries=0),
            schema=NanoAODSchema,
            post=post,
        )
        t1 = time.perf_counter()
        collector.extract_metrics_from_output(output)
        collector.set_coffea_report(report)

    collector.save_measurement(f"{output_manager.root_output_dir}/measurements")
    return {
        "label": label,
        "wall_time": t1 - t0,
        "output": output,
        "report": report,
        "metrics": collector.get_metrics(),
        "tracking_data": collector.tracking_data,
        "span_metrics": getattr(collector, "span_metrics", None),
    }

def haas_post_fn(procer, out):
    hists = defaultdict(dict)
    for ch, ch_histos in procer.analysis.nD_hists_per_region.items():
        for obs, obs_histo in ch_histos.items():
            hists[ch][obs] = obs_histo.snapshot(delete_from_server=True)
        
    out["histograms"] = hists

    return out


results = {}

with acquire_client(
    AF, close_after=AUTO_CLOSE_CLIENT, pip_packages=WORKER_DEPENDENCIES
, num_workers=n_workers, profile_suffix="full_run_haas_or_not", profile_output_dir=f"{output_manager.root_output_dir}/profiling/") as (client, cluster):
    print("▶ Running HaaS backend (HistServProcessor)…")
    haas_proc = HistServProcessor(
        config=validated_config,
        output_manager=output_manager,
        metadata_lookup=metadata_lookup,
    )
    results["haas"] = run_backend(client, haas_proc, "HaaS (histserv)", post=haas_post_fn)

    print("\n▶ Running reduce backend (HistLocalProcessor)…")
    local_proc = HistLocalProcessor(
        config=validated_config,
        output_manager=output_manager,
        metadata_lookup=metadata_lookup,
    )
    results["local"] = run_backend(client, local_proc, "local reduce")

for key, r in results.items():
    print(f"{r['label']}: {r['wall_time']:.1f}s, {r['output'].get('processed_events', 0):,} events")

## Metrics comparison

In [ ]:
from rich.console import Console
from roastcoffea.export.reporter import (
    format_chunk_metrics_table,
    format_throughput_table,
    format_event_processing_table,
    format_resources_table,
    format_timing_table,
)

console = Console()

for key in ("haas", "local"):
    r = results[key]
    console.rule(f"[bold]{r['label']}")
    print("📈 Throughput")
    console.print(format_throughput_table(r["metrics"]))
    print("⚡ Event processing")
    console.print(format_event_processing_table(r["metrics"]))
    print("🖥️  Resources")
    console.print(format_resources_table(r["metrics"]))
    print("⏱️  Timing")
    console.print(format_timing_table(r["metrics"]))
    print("Chunk Metrics (timing + memory per section)")
    chunk_table = format_chunk_metrics_table(r["metrics"])
    if chunk_table:
        console.print(chunk_table)
    else:
        print("No chunk metrics available")

### Side-by-side summary
A condensed view pulling the headline numbers from both backends.

In [ ]:
from rich.table import Table

haas = results["haas"]
local = results["local"]

def fmt(v, suffix=""):
    if v is None:
        return "—"
    if isinstance(v, float):
        return f"{v:,.2f}{suffix}"
    return f"{v:,}{suffix}"

rows = [
    ("Wall time (s)", haas["wall_time"], local["wall_time"]),
    ("Events processed",
     haas["output"].get("processed_events", 0),
     local["output"].get("processed_events", 0)),
    ("Throughput (events/s)",
     haas["output"].get("processed_events", 0) / haas["wall_time"],
     local["output"].get("processed_events", 0) / local["wall_time"]),
    ("Bytes read (MB)",
     haas["report"].get("bytesread", 0) / 1e6,
     local["report"].get("bytesread", 0) / 1e6),
    ("Chunks",
     haas["report"].get("chunks"),
     local["report"].get("chunks")),
]

table = Table(title="HaaS vs local summary")
table.add_column("Metric")
table.add_column("HaaS", justify="right")
table.add_column("Local", justify="right")
table.add_column("Δ (local − haas)", justify="right")
for name, h, l in rows:
    delta = (l - h) if (isinstance(h, (int, float)) and isinstance(l, (int, float))) else None
    table.add_row(name, fmt(h), fmt(l), fmt(delta))
console.print(table)

## Histogram correctness check

Written by Claude.

Pull the HaaS histogram back from the server via `.to_hist()` and compare bin contents to the locally-reduced histogram. Any divergence beyond float noise indicates a bug in one of the paths.

In [ ]:
import numpy as np                                                                                                                                                                    
                                                                                                                                                                                    
                                                                                                                                                                                    
def materialize(h):                                                                                                                                                                   
  """Return a hist.Hist regardless of ChunkedHist vs Hist."""                                                                                                                       
  return h.to_hist() if hasattr(h, "to_hist") else h
                                                                                                                                                                                    
                                                                                                                                                                                    
def compare(h_haas, h_local, atol=1e-6, rtol=0.0, max_bins_printed=10):                                                                                                               
  """Compare two histograms slice-by-slice along (process, variation).                                                                                                              
                                                                                                                                                                                    
  Returns a dict:
      {                                                                                                                                                                             
          "ok": bool,
          "messages": [str, ...],            # category-level issues
          "diffs": [                         # one entry per mismatching combo                                                                                                      
              {                                                                                                                                                                     
                  "process": str,                                                                                                                                                   
                  "variation": str,                                                                                                                                                 
                  "max_abs": float,
                  "sum_haas": float,                                                                                                                                                
                  "sum_local": float,
                  "bad_bins": [(idx, edge_lo, edge_hi, haas_val, local_val, abs_diff), ...],
              },                                                                                                                                                                    
              ...
          ],                                                                                                                                                                        
      }       
  """
  result = {"ok": True, "messages": [], "diffs": []}
                                                                                                                                                                                    
  # Observable edges must agree
  if list(h_haas.axes["observable"].edges) != list(h_local.axes["observable"].edges):                                                                                               
      result["ok"] = False                                                                                                                                                          
      result["messages"].append("observable edges differ")
      return result                                                                                                                                                                 
              
  edges = np.asarray(h_haas.axes["observable"].edges)

  haas_procs = set(h_haas.axes["process"])
  local_procs = set(h_local.axes["process"])
  if haas_procs != local_procs:                                                                                                                                                     
      result["ok"] = False
      result["messages"].append(                                                                                                                                                    
          f"process categories differ: "
          f"haas-only={sorted(haas_procs - local_procs)}, "
          f"local-only={sorted(local_procs - haas_procs)}"
      )

  haas_vars = set(h_haas.axes["variation"])
  local_vars = set(h_local.axes["variation"])
  if haas_vars != local_vars:
      result["ok"] = False
      result["messages"].append(
          f"variation categories differ: "
          f"haas-only={sorted(haas_vars - local_vars)}, "                                                                                                                           
          f"local-only={sorted(local_vars - haas_vars)}"
      )                                                                                                                                                                             
              
  for p in sorted(haas_procs & local_procs):
      for v in sorted(haas_vars & local_vars):
          a = h_haas[{"process": p, "variation": v}].view(flow=True)["value"]
          b = h_local[{"process": p, "variation": v}].view(flow=True)["value"]                                                                                                      
          abs_diff = np.abs(a - b)
          tol = atol + rtol * np.maximum(np.abs(a), np.abs(b))                                                                                                                      
          bad_mask = abs_diff > tol
                                                                                                                                                                                    
          if not np.any(bad_mask):
              continue                                                                                                                                                              
              
          result["ok"] = False                                                                                                                                                      
          # Skip flow bins when reporting (first = underflow, last = overflow)
          bad_indices = np.where(bad_mask)[0]                                                                                                                                       
          bad_bins = []                                                                                                                                                             
          for i in bad_indices[:max_bins_printed]:                                                                                                                                  
              # Map flow-included index i to physical bin edges                                                                                                                     
              if i == 0:                                                                                                                                                            
                  lo, hi = -np.inf, edges[0]                                                                                                                                        
              elif i == len(edges):                                                                                                                                                 
                  lo, hi = edges[-1], np.inf                                                                                                                                        
              else:                                                                                                                                                                 
                  lo, hi = edges[i - 1], edges[i]
              bad_bins.append((int(i), float(lo), float(hi),                                                                                                                        
                               float(a[i]), float(b[i]), float(abs_diff[i])))                                                                                                       

          result["diffs"].append({                                                                                                                                                  
              "process": p,
              "variation": v,                                                                                                                                                       
              "max_abs": float(abs_diff.max()),
              "n_bad_bins": int(bad_mask.sum()),
              "sum_haas": float(a.sum()),                                                                                                                                           
              "sum_local": float(b.sum()),
              "bad_bins": bad_bins,                                                                                                                                                 
          })  

  return result


haas_hists = results["haas"]["output"]["histograms"]
local_hists = results["local"]["output"]["histograms"]
                                                                                                                                                                                    
total_mismatches = 0
for channel, by_obs in haas_hists.items():                                                                                                                                            
  for obs, h_haas_raw in by_obs.items():
      h_local_raw = local_hists.get(channel, {}).get(obs)
      if h_local_raw is None:                                                                                                                                                       
          print(f"❌ {channel}/{obs}: missing in local output")
          total_mismatches += 1                                                                                                                                                     
          continue                                                                                                                                                                  

      h_haas = materialize(h_haas_raw)                                                                                                                                              
      h_local = materialize(h_local_raw)
      res = compare(h_haas, h_local)
                                                                                                                                                                                    
      status = "✅" if res["ok"] else "❌"
      print(f"{status} {channel}/{obs}")                                                                                                                                            
      for msg in res["messages"]:
          print(f"    {msg}")
      for d in res["diffs"]:
          print(f"    [{d['process']:<15} / {d['variation']:<20}] "
                f"max|Δ|={d['max_abs']:.3e}, {d['n_bad_bins']} bad bin(s), "                                                                                                        
                f"sum_haas={d['sum_haas']:.3f}, sum_local={d['sum_local']:.3f}")
          for idx, lo, hi, haas_v, local_v, diff in d["bad_bins"]:                                                                                                                  
              print(f"        bin {idx:3d} [{lo:8.2f}, {hi:8.2f}): "
                    f"haas={haas_v:12.4f}  local={local_v:12.4f}  Δ={diff:.3e}")                                                                                                    
      if not res["ok"]:                                                                                                                                                             
          total_mismatches += 1                                                                                                                                                     
                                                                                                                                                                                    
print(f"\n{total_mismatches} mismatch(es)")

In [ ]:
import numpy as np                                                                                                                                                                                                                       
import pandas as pd                  
                                                                                                                                                                                                                                       
haas_hists = results["haas"]["output"]["histograms"]                                                                                                                                                                                     
local_hists = results["local"]["output"]["histograms"]                                                                                                                                                                                   
                                                                                                                                                                                                                                       
rows = []                                                                                                                                                                                                                                
for channel, by_obs in haas_hists.items():                                                                                                                                                                                               
  for obs, h_haas_raw in by_obs.items():
      h_local_raw = local_hists.get(channel, {}).get(obs)
      if h_local_raw is None:                                                                                                                                                                                                          
          rows.append({
              "channel": channel, "observable": obs,                                                                                                                                                                                   
              "process": "—", "variation": "—",
              "sum_haas": np.nan, "sum_local": np.nan,                                                                                                                                                                                 
              "max_abs": np.nan, "status": "missing_local",
          })                                                                                                                                                                                                                           
          continue
                                                                                                                                                                                                                                       
      h_haas = materialize(h_haas_raw)
      h_local = materialize(h_local_raw)
      shared_procs = sorted(set(h_haas.axes["process"]) & set(h_local.axes["process"]))
      shared_vars = sorted(set(h_haas.axes["variation"]) & set(h_local.axes["variation"]))                                                                                                                                             
                                                                                                                                                                                                                                       
      for p in shared_procs:                                                                                                                                                                                                           
          for v in shared_vars:                                                                                                                                                                                                        
              a = h_haas[{"process": p, "variation": v}].view(flow=True)["value"]
              b = h_local[{"process": p, "variation": v}].view(flow=True)["value"]
              max_abs = float(np.max(np.abs(a - b))) if a.size else 0.0                                                                                                                                                                
              rows.append({                                                                                                                                                                                                            
                  "channel": channel,                                                                                                                                                                                                  
                  "observable": obs,                                                                                                                                                                                                   
                  "process": p,
                  "variation": v,
                  "sum_haas": float(a.sum()),
                  "sum_local": float(b.sum()),                                                                                                                                                                                         
                  "max_abs": max_abs,
                  "status": "ok" if max_abs < 1e-6 else "diff",                                                                                                                                                                        
              })                                                                                                                                                                                                                       

df = pd.DataFrame(rows)                                                                                                                                                                                                                  
df["rel_diff"] = (df["sum_haas"] - df["sum_local"]).abs() / df[["sum_haas", "sum_local"]].abs().max(axis=1).replace(0, np.nan)
                                                                                                                                                                                                                                       
# Display all rows                                                                                                                                                                                                                       
with pd.option_context("display.max_rows", None, "display.width", 200, "display.float_format", "{:.4g}".format):                                                                                                                         
  print(df.to_string(index=False))                                                                                                                                                                                                     
                                                                                                                                                                                                                                       
print()                                                                                                                                                                                                                                  
print(f"Rows:          {len(df)}")                                                                                                                                                                                                       
print(f"OK:            {(df['status'] == 'ok').sum()}")
print(f"Different:     {(df['status'] == 'diff').sum()}")                                                                                                                                                                                
print(f"Worst max|Δ|:  {df['max_abs'].max():.3e}")